In [18]:
import pandas as pd, numpy as np, matplotlib.pyplot as plt, seaborn as sns, glob, os
import scipy.io as sio

## Objectives
1. load QC neuron file
2. append columns for spikes, region, coordinates
3. concatenate across patients, and save

### variables

In [19]:
patient = 22

### 1. copy cluster-figs to new directory for easier QC
### 2. create corresponding preQC_df csv
note: clustID called unitID going forward


In [20]:
# 1. init new dir with only figs of clusters for easier QC
osort_CL_dir = f'../../results/2025{patient}/osort_mat/figs_clust'
os.makedirs(osort_CL_dir, exist_ok=True)

# 2. init df for pre-QC csv 
preQC_df = pd.DataFrame(columns=['chanID', 'unitID'])
chanIDs, unitIDs = [], []

# go through existing fig dir
osort_figs_dir = f'../../results/2025{patient}/osort_mat/figs/5'
for fig_file in glob.glob(f'{osort_figs_dir}/*CL*.png'):

    # grab only individual CL figs
    if not 'ALL' in os.path.basename(fig_file):
        
        # 1. copy individual CL figs to new dir
        dest = os.path.join(osort_CL_dir, os.path.basename(fig_file))
        if os.path.exists(dest): print(f'File already exists, skipping')
        else:
            print(f'Copying {fig_file} to {dest}')
            os.system(f'cp {fig_file} {osort_CL_dir}/')
            
        # 2. append IDs for preQC_df csv
        chanIDs.append(int(os.path.basename(fig_file).split('_')[0][1:]))
        unitIDs.append(int(os.path.basename(fig_file).split('_')[2]))
print('1. copied CL figs to new dir')

# create the preQC_df and save
preQC_df = pd.DataFrame({'chanID': chanIDs, 'unitID': unitIDs})
preQC_df = preQC_df.sort_values(by=['chanID', 'unitID']).reset_index(drop=True)

preQC_df_path = f'../../results/2025{patient}/records/preQC_pt{patient}.csv'
if os.path.exists(preQC_df_path):
    print(f'File already exists, skipping: {preQC_df_path}')
else:
    print(f'Saving preQC_df to {preQC_df_path}')
    preQC_df.to_csv(preQC_df_path, index=False)
print('2. created and saved preQC_df csv')
preQC_df

Copying ../../results/202522/osort_mat/figs/5/A207_CL_1918_THM_1.png to ../../results/202522/osort_mat/figs_clust/A207_CL_1918_THM_1.png
Copying ../../results/202522/osort_mat/figs/5/A193_CL_929_THM_1.png to ../../results/202522/osort_mat/figs_clust/A193_CL_929_THM_1.png
Copying ../../results/202522/osort_mat/figs/5/A208_CL_1336_THM_1.png to ../../results/202522/osort_mat/figs_clust/A208_CL_1336_THM_1.png
Copying ../../results/202522/osort_mat/figs/5/A206_CL_1532_THM_1.png to ../../results/202522/osort_mat/figs_clust/A206_CL_1532_THM_1.png
Copying ../../results/202522/osort_mat/figs/5/A198_CL_421_THM_1.png to ../../results/202522/osort_mat/figs_clust/A198_CL_421_THM_1.png
Copying ../../results/202522/osort_mat/figs/5/A202_CL_1173_THM_1.png to ../../results/202522/osort_mat/figs_clust/A202_CL_1173_THM_1.png
Copying ../../results/202522/osort_mat/figs/5/A195_CL_53_THM_1.png to ../../results/202522/osort_mat/figs_clust/A195_CL_53_THM_1.png
Copying ../../results/202522/osort_mat/figs/5/A20

,chanID,unitID
0,193,31
1,193,810
2,193,883
3,193,902
4,193,929
...,...,...
82,208,1284
83,208,1301
84,208,1329
85,208,1336


### after having performed manual QC, load QC_df:

In [21]:
QC_df = pd.read_csv(f'../../results/2025{patient}/records/QC_pt{patient}.csv')

# create list of dropped unitIDS
dropped_unitIDs = QC_df[QC_df['keep'] != 1]['unitID'].astype(int).tolist()
dropped_unitIDs.extend([0, 1, 99999999])

# drop unitIDs
QC_df = QC_df[~QC_df['unitID'].isin(dropped_unitIDs)].copy().reset_index(drop=True)
QC_df

,chanID,unitID,keep,notes
0,193,883,1.0,NaN
1,193,929,1.0,NaN
2,197,44,1.0,NaN
3,199,2436,1.0,FR fluctuation
4,199,2472,1.0,FR fluctuation
5,199,2477,1.0,FR fluctuation
6,200,889,1.0,NaN
7,201,2271,1.0,NaN
8,201,2283,1.0,NaN
9,204,1726,1.0,NaN


### helpers

In [22]:
def getunitID2spikes(unitIDs, spikes):
    ''' return dict with keys=unique units, and vals = list of corresponding spikes '''
    
    # initialize unit2spikes with keys as unique unit IDs and empty list as values
    unit2spikes = {}
    for unitID, spike in zip(unitIDs, spikes):

        if unitID in dropped_unitIDs: continue
        if unitID not in unit2spikes: unit2spikes[unitID] = [] # initialize

        unit2spikes[unitID].append(spike)

    return unit2spikes

### start creating neur/spikes df. First, add spike data from OSort mats.

In [25]:
samp_rate = 1000000
# columns
chanID_list, unitID_list, spikes_list, num_spikes_list, FR_list = [], [], [], [], []

# go through OSort mat files
for mat_file in glob.glob(f'../../results/2025{patient}/osort_mat/sorted_mats/5/*_sorted_new.mat'):

    # load channel mat
    chanMat = sio.loadmat(mat_file)
    chanID = int(os.path.basename(mat_file).split('_')[0][1:])

    # load vectors of spike times and corresponding units
    if chanMat['assignedNegative'].size == 0: continue
    spike_vector = chanMat['newTimestampsNegative'][0] # len = total n_spikes
    unit_vector = chanMat['assignedNegative'][0] # len = total n_spikes

    # create unit_vector => [spike_vector] for QCed units
    unit2spikes = getunitID2spikes(unit_vector, spike_vector)
    for unitID, unit_spike_list in unit2spikes.items():
        chanID_list.append(chanID)
        unitID_list.append(unitID)
        spikes = np.array(unit_spike_list) / samp_rate  # seconds
        spikes_list.append(spikes)
        num_spikes_list.append(len(spikes))
        FR_list.append(len(spikes) / (spikes[-1] - spikes[0]))

neur_df = pd.DataFrame({'chanID': chanID_list, 'unitID': unitID_list, 'spikes': spikes_list, 'num_spikes': num_spikes_list, 'FR': FR_list})
assert len(neur_df) == len(QC_df), f'Length mismatch: neur_df has {len(neur_df)} rows, QC_df has {len(QC_df)} rows'
neur_df


,chanID,unitID,spikes,num_spikes,FR
0,200,889,"[1.2737333333333334, 4.168333333333334, 5.4483...",3236,1.921668
1,201,2271,"[17.292166666666667, 17.33396666666667, 18.380...",4520,2.710302
2,201,2283,"[18.104666666666667, 25.1578, 32.396, 47.1109,...",2065,1.238960
3,197,44,"[17.3487, 28.755200000000002, 30.8730666666666...",440,0.263798
4,199,2477,"[0.3642666666666667, 0.38356666666666667, 0.39...",16289,9.665839
5,199,2436,"[6.560433333333334, 12.252566666666668, 47.828...",2996,1.784500
6,199,2472,"[7.637333333333334, 16.9942, 40.48683333333334...",1177,0.702765
7,193,929,"[1.4766333333333335, 1.5566666666666666, 1.716...",7246,4.303382
8,193,883,"[3.3423000000000003, 4.025933333333334, 4.1788...",2431,1.446764
9,204,1726,"[17.384533333333337, 17.67366666666667, 17.679...",2360,1.420181


### add region column by mapping channel -> region (label)

In [29]:
chanMap = sio.loadmat(glob.glob(f'../../results/2025{patient}/records/*ChannelMap*.mat')[0])

# variable across patients
if patient in [12, 21]: channelMap = chanMap['ChannelMap1'].flatten()
elif patient == 18: channelMap = chanMap['ChannelMap2'].flatten()

labelMap = chanMap['LabelMap'].flatten() # contains region labels
labelMap = np.array([str(label.squeeze()) for label in labelMap])  # convert to str

nan_mask = ~np.isnan(channelMap)
channel2label = dict(zip(channelMap[nan_mask], labelMap[nan_mask]))

neur_df['region'] = neur_df['chanID'].map(channel2label).fillna(neur_df['chanID']).apply(lambda x: str(x))
neur_df

IndexError: boolean index did not match indexed array along axis 0; size of axis is 272 but size of corresponding boolean axis is 234

### add coordinates columns by mapping region->coords

In [27]:
# cleaning function to handle nested arrays and bytes
def clean_entry(x):
    while isinstance(x, (np.ndarray, list)):
        x = x[0]
    if isinstance(x, (bytes, bytearray)):
        x = x.decode("utf-8", errors="ignore")
    return str(x)

at some point, figure out this code line by line

In [28]:
# load
electrodeInfo = sio.loadmat(glob.glob(
                f'../../results/2025{patient}/records/*DI_Electrodes*.mat')[0])
ElecMapRaw   = pd.DataFrame(electrodeInfo['ElecMapRaw']) # region -> ID
ElecXYZRaw   = pd.DataFrame(electrodeInfo['ElecXYZRaw']) # ID -> coordinates
ElecAtlasRaw = pd.DataFrame(electrodeInfo['ElecAtlasRaw']) # atlas coords?

atlas_index = 0 # NMM

# clean
region_s = ElecMapRaw[0].apply(clean_entry)                    # Series of regions
atlas_s  = ElecAtlasRaw.iloc[:, atlas_index].apply(clean_entry)  # Series of atlas regions

# build small tables
region2id_df = pd.DataFrame({"region": region_s.values,
                             "ID": np.arange(len(region_s))})
id2xyz_df = ElecXYZRaw.reset_index().rename(columns={'index':'ID', 0:'x', 1:'y', 2:'z'})
# xyz2atlasRegions = pd.DataFrame({"ID": np.arange(len(atlas_s)),
#                                  "atlas_region": atlas_s.values})

# merge
final_neur_df = (neur_df
                .merge(region2id_df, on='region', how='left')
                .merge(id2xyz_df, on='ID', how='left')
                # .merge(xyz2atlasRegions, on='ID', how='left')
                )
final_neur_df = final_neur_df.drop(columns=['ID'])
final_neur_df['spikes'] = final_neur_df['spikes'].apply(lambda x: np.array(x))  # ensure arrays


KeyError: 'region'

### inspect & save

In [30]:
print('example neuron')
print(f'last 5 spikes (s): {final_neur_df["spikes"].iloc[0][-5:]}')
print(f'last 5 spikes (min): {final_neur_df["spikes"].iloc[0][-5:]/60}')

# add patient as first column
if 'patient' not in final_neur_df: final_neur_df.insert(0, 'patient', patient)

final_neur_df

example neuron
last 5 spikes (s): [1737.31346667 1737.37963333 1738.62076667 1740.39936667 1740.55433333]
last 5 spikes (min): [28.95522444 28.95632722 28.97701278 29.00665611 29.00923889]


,patient,chanID,unitID,spikes,num_spikes,FR,region,x,y,z
0,21,219,1709,"[5.8342, 33.16630000000001, 34.1727, 34.288, 3...",2786,1.606023,mRAHIP8,21.000004,-1.29937,-15.600003
1,21,218,3888,"[0.3418666666666667, 0.6712, 1.758166666666666...",3976,2.241688,mRAHIP7,18.600004,-1.29937,-15.600003
2,21,218,3887,"[0.34526666666666667, 0.3496, 0.35416666666666...",15705,8.859160,mRAHIP7,18.600004,-1.29937,-15.600003
3,21,218,3845,"[186.2033, 189.2548666666667, 189.284366666666...",3452,2.174044,mRAHIP7,18.600004,-1.29937,-15.600003
4,21,220,2374,"[10.1639, 32.283433333333335, 35.6210666666666...",710,0.406395,220,NaN,NaN,NaN
5,21,220,2305,"[34.1908, 34.4417, 34.516533333333335, 34.5623...",13423,7.790637,220,NaN,NaN,NaN
6,21,222,878,"[0.05560000000000001, 0.7493333333333334, 0.82...",3460,1.949670,222,NaN,NaN,NaN
7,21,223,2489,"[0.047733333333333336, 0.1006, 0.1926, 0.2007,...",2159,1.216535,223,NaN,NaN,NaN
8,21,223,2492,"[2.8509333333333333, 3.1158333333333337, 3.505...",5723,3.261753,223,NaN,NaN,NaN
9,21,224,3097,"[5.0144, 5.237, 32.37466666666667, 34.28296666...",11870,6.796445,224,NaN,NaN,NaN


In [32]:
# make dir
processed_data_dir = f'../../results/2025{patient}/records/processed_data'
os.makedirs(processed_data_dir, exist_ok=True)

# save file
parquet_path = os.path.join(processed_data_dir, 'df_neurs.parquet')
if os.path.exists(parquet_path):     print(f'File already exists, skipping: {parquet_path}')
else:     final_neur_df.to_parquet(parquet_path, index=False)
# final_neur_df.to_parquet(parquet_path, index=False)

File already exists, skipping: ../../results/202522/records/processed_data/df_neurs.parquet
